# 01 — Data cleaning and EDA

Why this notebook exists: confirm the raw CSV matches the expected 7,043 rows,
check the known `TotalCharges` blank-string issue, and reproduce the SQL KPI
numbers independently before dashboarding.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

telco = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(telco.shape)
telco.head()

In [ ]:
# Data-quality checks: nulls here are only pandas NaN, so blank strings
# in TotalCharges need a separate explicit check below.
telco.info()
print(telco.isnull().sum())
print("duplicate customerID:", telco.duplicated("customerID").sum())
print(telco["Churn"].value_counts())
blank_total = (telco["TotalCharges"].astype(str).str.strip() == "").sum()
print("blank TotalCharges strings:", blank_total)

In [ ]:
# Cleaning mirrors sql/02: blanks become NaN (11 new-tenure rows),
# ChurnFlag gives a numeric target, TenureGroup matches the SQL bins,
# and revenue-at-risk stays a reporting proxy, not a model input.
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"].astype(str).str.strip(), errors="coerce")
telco["ChurnFlag"] = (telco["Churn"] == "Yes").astype(int)
tenure_bins = [-1, 6, 12, 24, 48, 72]
tenure_labels = ["0-6 Months", "7-12 Months", "13-24 Months", "25-48 Months", "49-72 Months"]
telco["TenureGroup"] = pd.cut(telco["tenure"], bins=tenure_bins, labels=tenure_labels)
telco["EstimatedRevenueAtRisk"] = np.where(telco["Churn"] == "Yes", telco["MonthlyCharges"], 0)
print(len(telco), int(telco["ChurnFlag"].sum()), round(telco["ChurnFlag"].mean() * 100, 2))

In [ ]:
telco["Churn"].value_counts().plot(kind="bar", color=["#2ca02c", "#d62728"])
plt.title("Churned vs retained")
plt.tight_layout()
plt.show()

In [ ]:
contract_churn = telco.groupby("Contract")["ChurnFlag"].mean().mul(100).sort_values(ascending=False)
sns.barplot(x=contract_churn.index, y=contract_churn.values, palette="Blues_d")
plt.title("Churn rate by contract")
plt.ylabel("%")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
tenure_order = ["0-6 Months", "7-12 Months", "13-24 Months", "25-48 Months", "49-72 Months"]
tenure_churn = telco.groupby("TenureGroup", observed=True)["ChurnFlag"].mean().mul(100).reindex(tenure_order)
sns.barplot(x=tenure_churn.index, y=tenure_churn.values, palette="Oranges_d")
plt.title("Churn by tenure")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
payment_churn = telco.groupby("PaymentMethod")["ChurnFlag"].mean().mul(100).sort_values(ascending=False)
sns.barplot(x=payment_churn.values, y=payment_churn.index, palette="Reds_d", orient="h")
plt.title("Churn by payment method")
plt.tight_layout()
plt.show()

In [ ]:
sns.histplot(data=telco, x="MonthlyCharges", hue="Churn", bins=30, kde=True, alpha=0.5)
plt.title("Monthly charges by churn")
plt.tight_layout()
plt.show()

In [ ]:
numeric_corr = telco[["tenure", "MonthlyCharges", "TotalCharges", "ChurnFlag"]].corr()
sns.heatmap(numeric_corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Numeric correlations")
plt.tight_layout()
plt.show()

In [ ]:
telco.to_csv("../data/processed/telco_customer_churn_cleaned.csv", index=False)
print("saved")